In [ ]:
# Cell 0: Optional Gemini intelligence layer
GEMINI_API_KEY = ""  # leave empty to skip
USE_GEMINI = bool(GEMINI_API_KEY)

if USE_GEMINI:
    import subprocess, sys, random
    subprocess.run([sys.executable, "-m", "pip", "install", "google-generativeai", "-q"])
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel("gemini-1.5-flash")

    def gemini_generate_training_pair(topic: str):
        prompt = f"""Generate one H:/ANRA: training pair for An-Ra.
Topic: {topic}
Output format exactly:
H: <question>
ANRA: <answer>"""
        try:
            text = gemini_model.generate_content(prompt).text.strip()
            if "H:" in text and "ANRA:" in text:
                h, a = text.split("\n", 1)
                return h.replace("H:", "").strip(), a.replace("ANRA:", "").strip()
        except Exception as e:
            print("Gemini error:", e)
        return None, None

    def gemini_evaluate_output(prompt: str, output: str) -> float:
        ep = f"Rate this answer from 0.0 to 1.0. Prompt: {prompt[:200]} Response: {output[:300]} Return only a float."
        try:
            return float(gemini_model.generate_content(ep).text.strip())
        except Exception:
            return 0.5

    def gemini_augment_dataset(existing_path: str, n_pairs: int = 50) -> str:
        from pathlib import Path
        import shutil
        topics = ["consciousness", "python", "debugging", "logic", "math", "identity", "systems"]
        pairs = []
        for _ in range(n_pairs):
            h, a = gemini_generate_training_pair(random.choice(topics))
            if h and a:
                pairs.append(f"H: {h}\nANRA: {a}")
        src = Path(existing_path)
        out = src.with_name(src.stem + "_gemini_augmented.txt")
        shutil.copy2(src, out)
        with open(out, "a", encoding="utf-8") as f:
            f.write("\n\n# Gemini-Augmented Training Data\n")
            f.write("\n\n".join(pairs))
        print(f"Saved {len(pairs)} pairs to {out}")
        return str(out)

    print("Gemini enabled")
else:
    print("Gemini disabled")
    gemini_generate_training_pair = None
    gemini_evaluate_output = None
    gemini_augment_dataset = None

In [ ]:
# Cell 1: Environment + Drive
import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('No GPU detected')

deps = [
    'sympy', 'sentence-transformers', 'fastapi', 'uvicorn',
    'peft', 'datasets', 'accelerate', 'httpx', 'psutil', 'scipy'
]
for dep in deps:
    try:
        __import__(dep.replace('-', '_').split('[')[0])
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', dep, '-q'])
        print('installed', dep)

DRIVE = Path('/content/drive/MyDrive/AnRa')
for d in ['checkpoints', 'identity', 'logs', 'sessions', 'memory_db', 'ouroboros']:
    (DRIVE / d).mkdir(parents=True, exist_ok=True)
print('Drive structure ready')

In [ ]:
# Cell 2: Repository Setup
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/YOURUSERNAME/An-Ra-the-new-AGI'  # update this
PROJECT_DIR = Path('/content/An-Ra-the-new-AGI-main')

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    os.system('git pull origin main')
    print('Repository updated')
else:
    zip_path = Path('/content/An-Ra-the-new-AGI-main.zip')
    if zip_path.exists():
        os.system(f'unzip -q {zip_path} -d /content/')
        print('Extracted from zip')
    else:
        os.system(f'git clone {REPO_URL} {PROJECT_DIR}')
        print('Repository cloned')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print('Path registry initialized:', PROJECT_DIR)

In [ ]:
# Cell 3: Full System Health Check
import json, sys, pickle
from pathlib import Path
from anra_paths import *

checks = {}

try:
    tok = pickle.load(open('tokenizer.pkl', 'rb'))
    checks['tokenizer'] = f"ok vocab={tok.vocab_size}"
except Exception as e:
    checks['tokenizer'] = f"fail {e}"

try:
    from anra_brain import CausalTransformer
    m = CausalTransformer(tok.vocab_size, 256, 4, 4, 128)
    checks['anra_brain_pytorch'] = f"ok params={sum(x.numel() for x in m.parameters())}"
except Exception as e:
    checks['anra_brain_pytorch'] = f"fail {e}"

try:
    sys.path.insert(0, str(CORE_DIR))
    from model import LanguageModel
    checks['core_model_numpy'] = 'ok'
except Exception as e:
    checks['core_model_numpy'] = f"warn {e}"

for name, mod in [
    ('phase3_identity', 'identity_injector'),
    ('phase3_ouroboros', 'ouroboros_numpy'),
    ('phase3_symbolic', 'symbolic_bridge'),
    ('phase3_sovereignty', 'sovereignty_bridge'),
]:
    try:
        __import__(mod)
        checks[name] = 'ok'
    except Exception as e:
        checks[name] = f"warn {e}"

try:
    from train_unified import main as _
    checks['train_unified'] = 'ok'
except Exception as e:
    checks['train_unified'] = f"fail {e}"

print(json.dumps(checks, indent=2))

In [ ]:
# Cell 4: Optional Gemini augmentation
N_GEMINI_PAIRS = 100

if USE_GEMINI and gemini_augment_dataset:
    dataset_path = gemini_augment_dataset('anra_dataset_v6_1.txt', N_GEMINI_PAIRS)
else:
    dataset_path = 'anra_dataset_v6_1.txt'
print('Dataset:', dataset_path)

In [ ]:
# Cell 5: Unified Training
TRAINING_MODE = 'identity'  # identity | ouroboros | full | eval
EPOCHS = 12
BATCH_SIZE = 32
LR = 3e-5
STEPS = 1200

import subprocess, sys
cmd = [
    sys.executable, 'train_unified.py',
    '--mode', TRAINING_MODE,
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--steps', str(STEPS),
]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)

In [ ]:
# Cell 6: Full test suite
import subprocess, sys
result = subprocess.run([sys.executable, 'test_suite.py'], capture_output=True, text=True, timeout=1200)
print(result.stdout[-5000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 7: Launch API (+ optional ngrok)
import subprocess, sys, threading, time

NGROK_TOKEN = ''  # optional

def _run_server():
    subprocess.run([sys.executable, 'app.py', '--port', '8000'])

threading.Thread(target=_run_server, daemon=True).start()
time.sleep(5)

if NGROK_TOKEN:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'])
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(8000)
    print('An-Ra live at:', url)
    print('Chat:', f'{url}/chat')
    print('Generate:', f'{url}/generate')
    print('Phase health:', f'{url}/phase-health')
    print('System map:', f'{url}/system-map')
    print('Sovereignty:', f'{url}/sovereignty-report')
else:
    print('Server at http://localhost:8000')